Image classification with neural Network model  with Pytorch
---
This Notebook show the whole process to buid MINST image classification from the holding of data to traning and testing of the model


In [16]:
import torch
from torch import nn
import torchvision
from torchvision.transforms import transforms
from torch.utils.data import DataLoader


In [17]:
# Dtata processeur
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(1.30, 0.328) ])

In [18]:
# downladingof the data

# training
data_train = torchvision.datasets.MNIST(root='/content/drive/MyDrive/Colab Notebooks/image_classification_project(MNIST)/data', train=True, download=True, transform=transform)

#test
data_test = torchvision.datasets.MNIST(root='/content/drive/MyDrive/Colab Notebooks/image_classification_project(MNIST)/data', train=False, download=True, transform=transform)

In [19]:
# loading of the Dataset:

#training
train = DataLoader(data_train,batch_size=64, shuffle=True)

#testing
test = DataLoader(data_test,batch_size=1000, shuffle=False)



In [20]:
# Model implementation
class MNISTCLasification(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten  =nn.Flatten() # it used to fla the image data because initialy , the image data are in 2D 28x28
        self.layers = nn.Sequential(
            nn.Linear(784,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [21]:
# set upo cuda device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [22]:
# model instanciation and turning in the device
model = MNISTCLasification()
model = model.to(device)

In [23]:
#traning of ours models
loss_function = nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(model.parameters(), lr=0.001)
def train_model(model, data, loss_function , optimizer):

    model.train()
    running_loss = 0.0
    correct=0
    total =0
    for batch_id,(data, target) in enumerate(data):
        data, target= data.to(device), target.to(device)
        optimizer.zero_grad()
        outpout = model(data)

        loss = loss_function(outpout, target)
        loss.backward()
        optimizer.step()
        # tracking progress of the training
        running_loss +=loss.item()
        _, predicted = outpout.max(1)
        # print(f'le outpout du model {outpout.max()}')
        total +=target.size(0)
        correct += predicted.eq(target).sum().item()
        if batch_id % 100 == 0 and batch_id >0:
            avg_loss = running_loss/100
            accuracy = 100*correct/total
            print(f'[{batch_id*64}/{60000}]',f'Loss!{avg_loss:.3f} | Accuracy: {accuracy:.1f}%')
            running_loss = 0.0


In [24]:
# model evaluation : test
def evaluate(model, data, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, target in data:
            inputs, target = inputs.to(device) , target.to(device)
            output = model(inputs)
            _, predicted = output.max(1)
            total+= target.size(0)
            correct += predicted.eq(target).sum().item()
        return 100*correct /total

In [25]:
#trainig loop
num_epoch = 10
for epoch in range(num_epoch):
    print(f"\nepoch: {epoch+1}")
    train_model(model,train,loss_function,optimizer)
    accuracy = evaluate(model, test,device)

    print(f'Test Accuracy :{accuracy:.2f}%')


epoch: 1
[6400/60000] Loss!1.793 | Accuracy: 47.2%
[12800/60000] Loss!0.884 | Accuracy: 61.3%
[19200/60000] Loss!0.631 | Accuracy: 68.2%
[25600/60000] Loss!0.531 | Accuracy: 72.4%
[32000/60000] Loss!0.466 | Accuracy: 75.2%
[38400/60000] Loss!0.460 | Accuracy: 77.0%
[44800/60000] Loss!0.444 | Accuracy: 78.4%
[51200/60000] Loss!0.456 | Accuracy: 79.3%
[57600/60000] Loss!0.436 | Accuracy: 80.1%
Test Accuracy :86.88%

epoch: 2
[6400/60000] Loss!0.437 | Accuracy: 87.4%
[12800/60000] Loss!0.399 | Accuracy: 87.7%
[19200/60000] Loss!0.399 | Accuracy: 87.7%
[25600/60000] Loss!0.378 | Accuracy: 87.9%
[32000/60000] Loss!0.408 | Accuracy: 87.9%
[38400/60000] Loss!0.362 | Accuracy: 88.2%
[44800/60000] Loss!0.365 | Accuracy: 88.3%
[51200/60000] Loss!0.379 | Accuracy: 88.3%
[57600/60000] Loss!0.363 | Accuracy: 88.4%
Test Accuracy :90.71%

epoch: 3
[6400/60000] Loss!0.370 | Accuracy: 89.4%
[12800/60000] Loss!0.402 | Accuracy: 88.6%
[19200/60000] Loss!0.354 | Accuracy: 88.9%
[25600/60000] Loss!0.338 |